# Ventas semanales

Comparación temporal: regresión Ridge de calendario y referencia estacional por sucursal/día. Horizonte de hasta 12 semanas; ventas y pedidos completados. El origen temporal se versiona con el modelo.

Las funciones de producción son la única implementación del entrenamiento. Estos experimentos registran artefactos en MLflow; no sustituyen la publicación activa. El DAG publica los siete módulos de forma atómica. No se consultan datos personales de contacto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'hgc-ml').exists())
sys.path.insert(0, str(ROOT / 'hgc-ml'))
from core.settings import postgres_engine
from training.common import setup_tracking
engine = postgres_engine()
setup_tracking()
with engine.connect() as connection:
    load_id = connection.execute(text('SELECT load_id FROM hgc_analytics.loads ORDER BY loaded_at DESC LIMIT 1')).scalar_one()
print('Snapshot:', load_id)


Snapshot: befab29d-54d0-4c88-aee6-0d59c6e120aa


In [2]:
frame=pd.read_sql_table('sys_sales_daily',engine,schema='hgc_analytics')
from training.sales import train_sales
sales=train_sales(frame,load_id)
sales['model']

2026/10/05 03:54:52 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 03:54:52 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


2026/10/05 03:54:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'hgc_sales' already exists. Creating a new version of this model...
Created version '7' of model 'hgc_sales'.


{'run_id': '11cf069865e94332ade176ea7d61be33',
 'algorithm': 'weekly_seasonal',
 'metrics': {'mae_revenue': 2335.9117542735044,
  'mae_orders': 9.426434676434676,
  'baseline_mae_revenue': 2335.9117542735044,
  'candidate_mae_revenue': 10790.78773848745},
 'cutoff': '2026-10-04',
 'feature_origin': '2015-05-15',
 'validation': 'Últimos 84 días; selección frente a promedio estacional',
 'limitations': 'Proyección desde el último dato. Banda diaria empírica del 90%, sin garantía de cobertura futura. No estima efectos causales de campañas.'}

In [3]:
from serving.predictors import infer
from serving.schemas import InferenceRequest
branch=int(frame.id_sucursal.iloc[0])
short=infer('sales',InferenceRequest(id_sucursal=branch,weeks=1))
long=infer('sales',InferenceRequest(id_sucursal=branch,weeks=4))
assert len(short['rows'])==7 and len(long['rows'])==28
pd.DataFrame([{'weeks':1,'forecast':short['value']},{'weeks':4,'forecast':long['value']}])

,weeks,forecast
0,1,101588.88
1,4,406355.52
